In [18]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 1: Imports
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

import warnings
warnings.filterwarnings("ignore")

print("Phase 2 environment ready.")

Phase 2 environment ready.


In [19]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 2: Load Phase 1 Customer Features
# ============================================================

from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Phase 1 processed data
FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_features.csv"
)

print("Loading:", FEATURE_FILE)

customer_features = pd.read_csv(FEATURE_FILE)

print("\nShape:", customer_features.shape)

print("\nColumns:")
print(customer_features.columns.tolist())

print("\nFirst 5 rows:")
display(customer_features.head())

Loading: c:\Users\akshi\Desktop\Behavioral-customer-segmentation\data\processed\customer_features.csv

Shape: (4335, 14)

Columns:
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'CustomerLifespanDays', 'AveragePurchaseIntervalDays', 'TotalQuantity', 'ProductDiversity', 'AverageBasketQuantity', 'AverageTransactionValue', 'SpendingTrend', 'FrequencyTrend', 'QuantityTrend', 'NewCustomer']

First 5 rows:


,CustomerID,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
0,12346.0,326,1,77183.60,0,54.025694,74215,1,74215.000000,77183.600000,-1.000000,-1.000000,-1.000000,0
1,12347.0,2,7,4310.00,365,60.839699,2458,103,351.142857,615.714286,0.112775,-0.125000,0.017390,0
2,12348.0,75,4,1437.24,282,94.250926,2332,21,583.000000,359.310000,-0.730133,-0.611111,-0.880907,0
3,12349.0,19,1,1457.55,0,54.025694,630,72,630.000000,1457.550000,0.000000,0.000000,0.000000,1
4,12350.0,310,1,294.40,0,54.025694,196,16,196.000000,294.400000,-1.000000,-1.000000,-1.000000,0


In [20]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 3: Load Clean Transaction Data
# ============================================================

TRANSACTION_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "clean_transactions.csv"
)

print("Loading:", TRANSACTION_FILE)

transactions = pd.read_csv(
    TRANSACTION_FILE,
    parse_dates=["InvoiceDate"]
)

print("\nShape:", transactions.shape)

print("\nColumns:")
print(transactions.columns.tolist())

print("\nDate range:")
print(transactions["InvoiceDate"].min())
print("to")
print(transactions["InvoiceDate"].max())

print("\nUnique customers:", transactions["CustomerID"].nunique())
print("Unique invoices:", transactions["InvoiceNo"].nunique())

display(transactions.head())

Loading: c:\Users\akshi\Desktop\Behavioral-customer-segmentation\data\processed\clean_transactions.csv

Shape: (391298, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Date range:
2010-12-01 08:26:00
to
2011-12-09 12:50:00

Unique customers: 4335
Unique invoices: 18416


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [21]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 4: Historical vs Recent Periods
# ============================================================

# Reference date used in Phase 1
reference_date = transactions["InvoiceDate"].max() + pd.Timedelta(days=1)

# Split date
split_date = pd.Timestamp("2011-07-01")

# Historical period
historical_transactions = transactions[
    transactions["InvoiceDate"] < split_date
].copy()

# Recent period
recent_transactions = transactions[
    transactions["InvoiceDate"] >= split_date
].copy()

print("Reference date:", reference_date)
print("Split date:", split_date)

print("\n--- HISTORICAL PERIOD ---")
print("Date range:")
print(historical_transactions["InvoiceDate"].min())
print("to")
print(historical_transactions["InvoiceDate"].max())

print("Transactions:", len(historical_transactions))
print("Customers:", historical_transactions["CustomerID"].nunique())
print("Quantity:", historical_transactions["Quantity"].sum())
print(
    "Spending:",
    (
        historical_transactions["Quantity"]
        * historical_transactions["UnitPrice"]
    ).sum()
)

print("\n--- RECENT PERIOD ---")
print("Date range:")
print(recent_transactions["InvoiceDate"].min())
print("to")
print(recent_transactions["InvoiceDate"].max())

print("Transactions:", len(recent_transactions))
print("Customers:", recent_transactions["CustomerID"].nunique())
print("Quantity:", recent_transactions["Quantity"].sum())
print(
    "Spending:",
    (
        recent_transactions["Quantity"]
        * recent_transactions["UnitPrice"]
    ).sum()
)

Reference date: 2011-12-10 12:50:00
Split date: 2011-07-01 00:00:00

--- HISTORICAL PERIOD ---
Date range:
2010-12-01 08:26:00
to
2011-06-30 20:08:00
Transactions: 170034
Customers: 2959
Quantity: 2297323
Spending: 3923314.201

--- RECENT PERIOD ---
Date range:
2011-07-01 08:16:00
to
2011-12-09 12:50:00
Transactions: 221264
Customers: 3361
Quantity: 2844610
Spending: 4820764.443


In [22]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 5: Customer-Level Historical/Recent Behavior
# ============================================================

def build_period_features(df, period_name):
    """Create monthly-normalized customer behavior for one period."""

    result = (
        df.groupby("CustomerID")
        .agg(
            Spending=(
                "InvoiceNo",
                lambda x: 0  # temporary placeholder
            )
        )
        .reset_index()
    )

    # Transaction value
    df = df.copy()
    df["TransactionValue"] = df["Quantity"] * df["UnitPrice"]

    result = (
        df.groupby("CustomerID")
        .agg(
            Spending=("TransactionValue", "sum"),
            Frequency=("InvoiceNo", "nunique"),
            Quantity=("Quantity", "sum")
        )
        .reset_index()
    )

    # Normalize according to the period length
    if period_name == "Historical":
        months = 7
    else:
        months = 6

    result["MonthlySpending"] = result["Spending"] / months
    result["MonthlyFrequency"] = result["Frequency"] / months
    result["MonthlyQuantity"] = result["Quantity"] / months

    result["Period"] = period_name

    return result[
        [
            "CustomerID",
            "Period",
            "MonthlySpending",
            "MonthlyFrequency",
            "MonthlyQuantity"
        ]
    ]


# Build both periods
historical_period = build_period_features(
    historical_transactions,
    "Historical"
)

recent_period = build_period_features(
    recent_transactions,
    "Recent"
)

# Combine
customer_period = pd.concat(
    [historical_period, recent_period],
    ignore_index=True
)

print("customer_period shape:", customer_period.shape)

print("\nRows per period:")
print(customer_period["Period"].value_counts())

print("\nCustomers per period:")
print(
    customer_period.groupby("Period")["CustomerID"]
    .nunique()
)

display(customer_period.head(10))

customer_period shape: (6320, 5)

Rows per period:
Period
Recent        3361
Historical    2959
Name: count, dtype: int64

Customers per period:
Period
Historical    2959
Recent        3361
Name: CustomerID, dtype: int64


,CustomerID,Period,MonthlySpending,MonthlyFrequency,MonthlyQuantity
0,12346.0,Historical,11026.228571,0.142857,10602.142857
1,12347.0,Historical,315.135714,0.571429,187.571429
2,12348.0,Historical,166.748571,0.428571,302.285714
3,12350.0,Historical,42.057143,0.142857,28.000000
4,12352.0,Historical,91.644286,0.571429,35.571429
5,12353.0,Historical,12.714286,0.142857,2.857143
6,12354.0,Historical,154.200000,0.142857,75.714286
7,12355.0,Historical,65.628571,0.142857,34.285714
8,12356.0,Historical,347.011429,0.285714,224.000000
9,12359.0,Historical,499.390000,0.428571,137.857143


In [23]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 6: Merge Historical and Recent Behavior
# ============================================================

historical = (
    customer_period[
        customer_period["Period"] == "Historical"
    ][
        [
            "CustomerID",
            "MonthlySpending",
            "MonthlyFrequency",
            "MonthlyQuantity"
        ]
    ]
    .rename(columns={
        "MonthlySpending": "HistoricalMonthlySpending",
        "MonthlyFrequency": "HistoricalMonthlyFrequency",
        "MonthlyQuantity": "HistoricalMonthlyQuantity"
    })
)

recent = (
    customer_period[
        customer_period["Period"] == "Recent"
    ][
        [
            "CustomerID",
            "MonthlySpending",
            "MonthlyFrequency",
            "MonthlyQuantity"
        ]
    ]
    .rename(columns={
        "MonthlySpending": "RecentMonthlySpending",
        "MonthlyFrequency": "RecentMonthlyFrequency",
        "MonthlyQuantity": "RecentMonthlyQuantity"
    })
)

# Outer merge so that new and inactive customers are retained
customer_trends = historical.merge(
    recent,
    on="CustomerID",
    how="outer"
)

# Missing activity means no activity in that period
period_columns = [
    "HistoricalMonthlySpending",
    "HistoricalMonthlyFrequency",
    "HistoricalMonthlyQuantity",
    "RecentMonthlySpending",
    "RecentMonthlyFrequency",
    "RecentMonthlyQuantity"
]

customer_trends[period_columns] = (
    customer_trends[period_columns]
    .fillna(0)
)

print("customer_trends shape:", customer_trends.shape)

print("\nMissing values:")
print(customer_trends[period_columns].isna().sum())

print("\nCustomers with historical activity:",
      (customer_trends["HistoricalMonthlyFrequency"] > 0).sum())

print("Customers with recent activity:",
      (customer_trends["RecentMonthlyFrequency"] > 0).sum())

print("\nFirst 10 customers:")
display(customer_trends.head(10))

customer_trends shape: (4335, 7)

Missing values:
HistoricalMonthlySpending     0
HistoricalMonthlyFrequency    0
HistoricalMonthlyQuantity     0
RecentMonthlySpending         0
RecentMonthlyFrequency        0
RecentMonthlyQuantity         0
dtype: int64

Customers with historical activity: 2959
Customers with recent activity: 3361

First 10 customers:


,CustomerID,HistoricalMonthlySpending,HistoricalMonthlyFrequency,HistoricalMonthlyQuantity,RecentMonthlySpending,RecentMonthlyFrequency,RecentMonthlyQuantity
0,12346.0,11026.228571,0.142857,10602.142857,0.000000,0.000000,0.000000
1,12347.0,315.135714,0.571429,187.571429,350.675000,0.500000,190.833333
2,12348.0,166.748571,0.428571,302.285714,45.000000,0.166667,36.000000
3,12349.0,0.000000,0.000000,0.000000,242.925000,0.166667,105.000000
4,12350.0,42.057143,0.142857,28.000000,0.000000,0.000000,0.000000
5,12352.0,91.644286,0.571429,35.571429,124.038333,0.500000,46.166667
6,12353.0,12.714286,0.142857,2.857143,0.000000,0.000000,0.000000
7,12354.0,154.200000,0.142857,75.714286,0.000000,0.000000,0.000000
8,12355.0,65.628571,0.142857,34.285714,0.000000,0.000000,0.000000
9,12356.0,347.011429,0.285714,224.000000,9.725000,0.166667,0.833333


In [24]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 7: Behavioral Feature Engineering
# ============================================================

phase2 = customer_trends.copy()

# ------------------------------------------------------------
# 1. Absolute behavioral changes
# ------------------------------------------------------------

phase2["SpendingChange"] = (
    phase2["RecentMonthlySpending"]
    - phase2["HistoricalMonthlySpending"]
)

phase2["FrequencyChange"] = (
    phase2["RecentMonthlyFrequency"]
    - phase2["HistoricalMonthlyFrequency"]
)

phase2["QuantityChange"] = (
    phase2["RecentMonthlyQuantity"]
    - phase2["HistoricalMonthlyQuantity"]
)

# ------------------------------------------------------------
# 2. Recent activity share
# ------------------------------------------------------------

spending_total = (
    phase2["HistoricalMonthlySpending"]
    + phase2["RecentMonthlySpending"]
)

frequency_total = (
    phase2["HistoricalMonthlyFrequency"]
    + phase2["RecentMonthlyFrequency"]
)

quantity_total = (
    phase2["HistoricalMonthlyQuantity"]
    + phase2["RecentMonthlyQuantity"]
)

phase2["RecentSpendingShare"] = np.where(
    spending_total > 0,
    phase2["RecentMonthlySpending"] / spending_total,
    0
)

phase2["RecentFrequencyShare"] = np.where(
    frequency_total > 0,
    phase2["RecentMonthlyFrequency"] / frequency_total,
    0
)

phase2["RecentQuantityShare"] = np.where(
    quantity_total > 0,
    phase2["RecentMonthlyQuantity"] / quantity_total,
    0
)

# ------------------------------------------------------------
# 3. Activity consistency
# ------------------------------------------------------------

phase2["ActivityConsistency"] = (
    phase2[
        [
            "RecentMonthlyFrequency",
            "HistoricalMonthlyFrequency"
        ]
    ].min(axis=1)
    /
    phase2[
        [
            "RecentMonthlyFrequency",
            "HistoricalMonthlyFrequency"
        ]
    ].max(axis=1).replace(0, np.nan)
).fillna(0)

# ------------------------------------------------------------
# 4. New customer indicator
# ------------------------------------------------------------

phase2["NewCustomer"] = (
    (
        (phase2["HistoricalMonthlyFrequency"] == 0)
        &
        (phase2["RecentMonthlyFrequency"] > 0)
    )
    .astype(int)
)

# ------------------------------------------------------------
# 5. List new features
# ------------------------------------------------------------

new_phase2_features = [
    "SpendingChange",
    "FrequencyChange",
    "QuantityChange",
    "RecentSpendingShare",
    "RecentFrequencyShare",
    "RecentQuantityShare",
    "ActivityConsistency",
    "NewCustomer"
]

print("Phase 2 feature matrix shape:", phase2.shape)

print("\nNew Phase 2 features:")
for feature in new_phase2_features:
    print(" -", feature)

print("\nMissing values:")
print(phase2[new_phase2_features].isna().sum())

print("\nFeature summary:")
display(
    phase2[new_phase2_features].describe().T
)

Phase 2 feature matrix shape: (4335, 15)

New Phase 2 features:
 - SpendingChange
 - FrequencyChange
 - QuantityChange
 - RecentSpendingShare
 - RecentFrequencyShare
 - RecentQuantityShare
 - ActivityConsistency
 - NewCustomer

Missing values:
SpendingChange          0
FrequencyChange         0
QuantityChange          0
RecentSpendingShare     0
RecentFrequencyShare    0
RecentQuantityShare     0
ActivityConsistency     0
NewCustomer             0
dtype: int64

Feature summary:


,count,mean,std,min,25%,50%,75%,max
SpendingChange,4335.0,56.052430,625.284654,-11026.228571,-32.293690,22.250000,82.576548,28077.852381
FrequencyChange,4335.0,0.082633,0.408091,-4.857143,-0.142857,0.071429,0.214286,9.642857
QuantityChange,4335.0,33.659208,335.556460,-10602.142857,-15.714286,13.166667,53.559524,13498.880952
RecentSpendingShare,4335.0,0.566508,0.384192,0.000000,0.237584,0.593581,1.000000,1.000000
RecentFrequencyShare,4335.0,0.563504,0.378668,0.000000,0.280000,0.538462,1.000000,1.000000
RecentQuantityShare,4335.0,0.572827,0.387178,0.000000,0.216752,0.619710,1.000000,1.000000
ActivityConsistency,4335.0,0.286017,0.347692,0.000000,0.000000,0.000000,0.583333,1.000000
NewCustomer,4335.0,0.317416,0.465525,0.000000,0.000000,0.000000,1.000000,1.000000


In [25]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 8: Robust Behavioral Change Features
# ============================================================

phase2["SpendingChangeLog"] = (
    np.sign(phase2["SpendingChange"])
    * np.log1p(np.abs(phase2["SpendingChange"]))
)

phase2["QuantityChangeLog"] = (
    np.sign(phase2["QuantityChange"])
    * np.log1p(np.abs(phase2["QuantityChange"]))
)

phase2["FrequencyChangeLog"] = (
    np.sign(phase2["FrequencyChange"])
    * np.log1p(np.abs(phase2["FrequencyChange"]))
)

robust_change_features = [
    "SpendingChangeLog",
    "FrequencyChangeLog",
    "QuantityChangeLog"
]

print("Robust change feature summary:")

display(
    phase2[robust_change_features].describe().T
)

print("\nCheck for missing values:")
print(
    phase2[robust_change_features]
    .isna()
    .sum()
)

Robust change feature summary:


,count,mean,std,min,25%,50%,75%,max
SpendingChangeLog,4335.0,0.979172,4.083676,-9.308123,-3.505368,3.146305,4.425762,10.242772
FrequencyChangeLog,4335.0,0.062419,0.276825,-1.767662,-0.133531,0.068993,0.194156,2.364889
QuantityChangeLog,4335.0,1.006861,3.581693,-9.268906,-2.816264,2.650892,3.999292,9.510436



Check for missing values:
SpendingChangeLog     0
FrequencyChangeLog    0
QuantityChangeLog     0
dtype: int64


In [27]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 9: Feature Correlation Analysis
# ============================================================

# Phase 1 features
phase1_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "TotalQuantity",
    "ProductDiversity",
    "AverageBasketQuantity",
    "AverageTransactionValue",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

# Phase 2 behavioral features
phase2_features = [
    "SpendingChangeLog",
    "FrequencyChangeLog",
    "QuantityChangeLog",
    "RecentSpendingShare",
    "RecentFrequencyShare",
    "RecentQuantityShare",
    "ActivityConsistency"
]

# ------------------------------------------------------------
# Build one clean feature table
# ------------------------------------------------------------

correlation_data = pd.concat(
    [
        customer_features[phase1_features].reset_index(drop=True),
        phase2[phase2_features].reset_index(drop=True)
    ],
    axis=1
)

print("Correlation dataset shape:", correlation_data.shape)

# ------------------------------------------------------------
# Correlation matrix
# ------------------------------------------------------------

corr = correlation_data.corr()

print("\nCorrelations of Phase 2 features with Phase 1 features:")

display(
    corr.loc[
        phase2_features,
        phase1_features
    ].round(3)
)

Correlation dataset shape: (4335, 20)

Correlations of Phase 2 features with Phase 1 features:


,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
SpendingChangeLog,-0.605,0.084,0.087,0.023,-0.065,0.074,0.120,0.004,0.016,0.035,0.526,0.036,0.545
FrequencyChangeLog,-0.489,0.119,0.139,-0.022,-0.123,0.107,0.162,0.004,0.005,-0.001,0.620,-0.001,0.459
QuantityChangeLog,-0.591,0.083,0.077,0.035,-0.064,0.060,0.142,0.000,0.013,0.037,0.512,0.038,0.519
RecentSpendingShare,-0.716,-0.003,0.007,-0.054,-0.069,0.006,0.050,0.005,0.007,0.018,0.454,0.018,0.770
RecentFrequencyShare,-0.713,-0.017,-0.004,-0.076,-0.081,-0.005,0.036,-0.004,-0.007,-0.000,0.476,0.000,0.786
RecentQuantityShare,-0.717,0.002,0.007,-0.039,-0.060,0.005,0.056,0.004,0.005,0.018,0.457,0.018,0.752
ActivityConsistency,-0.411,0.347,0.159,0.801,0.411,0.169,0.326,0.010,0.017,0.025,0.176,0.026,-0.561


In [28]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 10: New Feature-to-Feature Correlation
# ============================================================

new_feature_corr = (
    phase2[phase2_features]
    .corr()
)

print("Correlation among new Phase 2 features:")

display(
    new_feature_corr.round(3)
)

Correlation among new Phase 2 features:


,SpendingChangeLog,FrequencyChangeLog,QuantityChangeLog,RecentSpendingShare,RecentFrequencyShare,RecentQuantityShare,ActivityConsistency
SpendingChangeLog,1.000,0.772,0.926,0.866,0.813,0.855,-0.015
FrequencyChangeLog,0.772,1.000,0.748,0.697,0.725,0.690,-0.094
QuantityChangeLog,0.926,0.748,1.000,0.830,0.783,0.854,-0.000
RecentSpendingShare,0.866,0.697,0.830,1.000,0.977,0.989,-0.077
RecentFrequencyShare,0.813,0.725,0.783,0.977,1.000,0.968,-0.091
RecentQuantityShare,0.855,0.690,0.854,0.989,0.968,1.000,-0.059
ActivityConsistency,-0.015,-0.094,-0.000,-0.077,-0.091,-0.059,1.000


In [29]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 11: Candidate Behavioral Feature Set
# ============================================================

phase2_candidate_features = [
    "SpendingChangeLog",
    "FrequencyChangeLog",
    "RecentSpendingShare",
    "ActivityConsistency",
    "NewCustomer"
]

phase2_candidate = phase2[
    phase2_candidate_features
].copy()

print("Phase 2 candidate shape:", phase2_candidate.shape)

print("\nFeatures:")
for feature in phase2_candidate_features:
    print(" -", feature)

print("\nMissing values:")
print(phase2_candidate.isna().sum())

print("\nSummary:")
display(
    phase2_candidate.describe().T
)

Phase 2 candidate shape: (4335, 5)

Features:
 - SpendingChangeLog
 - FrequencyChangeLog
 - RecentSpendingShare
 - ActivityConsistency
 - NewCustomer

Missing values:
SpendingChangeLog      0
FrequencyChangeLog     0
RecentSpendingShare    0
ActivityConsistency    0
NewCustomer            0
dtype: int64

Summary:


,count,mean,std,min,25%,50%,75%,max
SpendingChangeLog,4335.0,0.979172,4.083676,-9.308123,-3.505368,3.146305,4.425762,10.242772
FrequencyChangeLog,4335.0,0.062419,0.276825,-1.767662,-0.133531,0.068993,0.194156,2.364889
RecentSpendingShare,4335.0,0.566508,0.384192,0.000000,0.237584,0.593581,1.000000,1.000000
ActivityConsistency,4335.0,0.286017,0.347692,0.000000,0.000000,0.000000,0.583333,1.000000
NewCustomer,4335.0,0.317416,0.465525,0.000000,0.000000,0.000000,1.000000,1.000000


In [30]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 12: Experimental Feature Sets
# ============================================================

# ------------------------------------------------------------
# 1. RFM baseline
# ------------------------------------------------------------

rfm_features = [
    "Recency",
    "Frequency",
    "Monetary"
]

# ------------------------------------------------------------
# 2. RFM + behavioral
# ------------------------------------------------------------

rfm_behavioral_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "SpendingChangeLog",
    "FrequencyChangeLog"
]

# ------------------------------------------------------------
# 3. RFM + temporal / activity
# ------------------------------------------------------------

rfm_temporal_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "RecentSpendingShare",
    "ActivityConsistency",
    "NewCustomer"
]

# ------------------------------------------------------------
# 4. Full Phase 1 feature representation
# ------------------------------------------------------------

phase1_full_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "TotalQuantity",
    "ProductDiversity",
    "AverageBasketQuantity",
    "AverageTransactionValue",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

# ------------------------------------------------------------
# 5. Enhanced Phase 2 representation
# ------------------------------------------------------------

phase2_enhanced_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "TotalQuantity",
    "ProductDiversity",
    "AverageBasketQuantity",
    "AverageTransactionValue",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer",
    "SpendingChangeLog",
    "FrequencyChangeLog",
    "RecentSpendingShare",
    "ActivityConsistency"
]

# ------------------------------------------------------------
# Store all experiments
# ------------------------------------------------------------

feature_sets = {
    "RFM": rfm_features,
    "RFM_Behavioral": rfm_behavioral_features,
    "RFM_Temporal": rfm_temporal_features,
    "Phase1_Full": phase1_full_features,
    "Phase2_Enhanced": phase2_enhanced_features
}

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("Phase 2 experimental feature sets:\n")

for name, features in feature_sets.items():
    print(f"{name}: {len(features)} features")
    print(features)
    print()

Phase 2 experimental feature sets:

RFM: 3 features
['Recency', 'Frequency', 'Monetary']

RFM_Behavioral: 5 features
['Recency', 'Frequency', 'Monetary', 'SpendingChangeLog', 'FrequencyChangeLog']

RFM_Temporal: 6 features
['Recency', 'Frequency', 'Monetary', 'RecentSpendingShare', 'ActivityConsistency', 'NewCustomer']

Phase1_Full: 13 features
['Recency', 'Frequency', 'Monetary', 'CustomerLifespanDays', 'AveragePurchaseIntervalDays', 'TotalQuantity', 'ProductDiversity', 'AverageBasketQuantity', 'AverageTransactionValue', 'SpendingTrend', 'FrequencyTrend', 'QuantityTrend', 'NewCustomer']

Phase2_Enhanced: 17 features
['Recency', 'Frequency', 'Monetary', 'CustomerLifespanDays', 'AveragePurchaseIntervalDays', 'TotalQuantity', 'ProductDiversity', 'AverageBasketQuantity', 'AverageTransactionValue', 'SpendingTrend', 'FrequencyTrend', 'QuantityTrend', 'NewCustomer', 'SpendingChangeLog', 'FrequencyChangeLog', 'RecentSpendingShare', 'ActivityConsistency']



In [32]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 13: Transformation + Scaling
# ============================================================

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Create ONE clean master feature table
# ------------------------------------------------------------

master_features = customer_features.copy()

# Add only the genuinely new Phase 2 features
for feature in [
    "SpendingChangeLog",
    "FrequencyChangeLog",
    "RecentSpendingShare",
    "ActivityConsistency"
]:
    master_features[feature] = phase2[feature].values

print("Master feature table shape:", master_features.shape)

# ------------------------------------------------------------
# Features requiring log1p transformation
# ------------------------------------------------------------

positive_skewed_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "TotalQuantity",
    "ProductDiversity",
    "AverageBasketQuantity",
    "AverageTransactionValue"
]

# ------------------------------------------------------------
# Transform + scale function
# ------------------------------------------------------------

def prepare_features(feature_list, data):

    transformed = data[feature_list].copy()

    for feature in transformed.columns:

        if feature in positive_skewed_features:

            # All these variables are non-negative
            transformed[feature] = np.log1p(
                transformed[feature]
            )

    scaler = StandardScaler()

    scaled_array = scaler.fit_transform(transformed)

    scaled_df = pd.DataFrame(
        scaled_array,
        columns=feature_list,
        index=data.index
    )

    return transformed, scaled_df, scaler


# ------------------------------------------------------------
# Prepare all experimental feature sets
# ------------------------------------------------------------

prepared_feature_sets = {}
scalers = {}

for name, features in feature_sets.items():

    transformed, scaled, scaler = prepare_features(
        features,
        master_features
    )

    prepared_feature_sets[name] = scaled
    scalers[name] = scaler

    print(
        f"{name}: "
        f"original={len(features)}, "
        f"scaled shape={scaled.shape}"
    )

print("\nAll feature sets prepared successfully.")

Master feature table shape: (4335, 18)
RFM: original=3, scaled shape=(4335, 3)
RFM_Behavioral: original=5, scaled shape=(4335, 5)
RFM_Temporal: original=6, scaled shape=(4335, 6)
Phase1_Full: original=13, scaled shape=(4335, 13)
Phase2_Enhanced: original=17, scaled shape=(4335, 17)

All feature sets prepared successfully.


In [33]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 14: PCA Variance Analysis
# ============================================================

pca_results = []

pca_models = {}

for name, scaled_data in prepared_feature_sets.items():

    # Fit PCA with all available components
    pca_full = PCA()
    pca_full.fit(scaled_data)

    cumulative_variance = np.cumsum(
        pca_full.explained_variance_ratio_
    )

    # Components needed for each variance target
    components_90 = (
        np.argmax(cumulative_variance >= 0.90) + 1
    )

    components_95 = (
        np.argmax(cumulative_variance >= 0.95) + 1
    )

    components_99 = (
        np.argmax(cumulative_variance >= 0.99) + 1
    )

    pca_results.append({
        "FeatureSet": name,
        "OriginalFeatures": scaled_data.shape[1],
        "PCs_90": components_90,
        "PCs_95": components_95,
        "PCs_99": components_99,
        "Variance_90": cumulative_variance[components_90 - 1],
        "Variance_95": cumulative_variance[components_95 - 1],
        "Variance_99": cumulative_variance[components_99 - 1]
    })

    pca_models[name] = pca_full

pca_results_df = pd.DataFrame(pca_results)

print("PCA variance analysis:")
display(
    pca_results_df.round(4)
)


PCA variance analysis:


,FeatureSet,OriginalFeatures,PCs_90,PCs_95,PCs_99,Variance_90,Variance_95,Variance_99
0,RFM,3,2,3,3,0.9385,1.0000,1.0000
1,RFM_Behavioral,5,3,4,5,0.9206,0.9674,1.0000
2,RFM_Temporal,6,3,4,6,0.9092,0.9617,1.0000
3,Phase1_Full,13,6,7,9,0.9257,0.9574,0.9926
4,Phase2_Enhanced,17,7,9,12,0.9201,0.9554,0.9934


In [34]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 15: Create PCA Representations
# ============================================================

pca_target = 0.90

pca_feature_sets = {}
pca_explained = {}

for name, scaled_data in prepared_feature_sets.items():

    # Fit PCA
    pca = PCA(
        n_components=pca_target,
        svd_solver="full"
    )

    transformed = pca.fit_transform(scaled_data)

    pca_feature_sets[name] = transformed

    pca_explained[name] = {
        "components": transformed.shape[1],
        "explained_variance": pca.explained_variance_ratio_.sum()
    }

    print(
        f"{name}: "
        f"{transformed.shape[1]} PCs | "
        f"variance={pca.explained_variance_ratio_.sum():.4f}"
    )

print("\nPCA representations created successfully.")

RFM: 2 PCs | variance=0.9385
RFM_Behavioral: 3 PCs | variance=0.9206
RFM_Temporal: 3 PCs | variance=0.9092
Phase1_Full: 6 PCs | variance=0.9257
Phase2_Enhanced: 7 PCs | variance=0.9201

PCA representations created successfully.


In [35]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 16: Standard K-Means Screening
# ============================================================

k_values = range(3, 11)

kmeans_results = []

for feature_set_name, X_pca in pca_feature_sets.items():

    print(f"\nTesting: {feature_set_name}")

    for k in k_values:

        kmeans = KMeans(
            n_clusters=k,
            random_state=82,
            n_init=20,
            max_iter=300
        )

        labels = kmeans.fit_predict(X_pca)

        # Cluster sizes
        cluster_counts = np.bincount(labels)

        min_cluster_size = cluster_counts.min()
        min_cluster_pct = (
            min_cluster_size / len(labels) * 100
        )

        # Metrics
        silhouette = silhouette_score(
            X_pca,
            labels
        )

        davies_bouldin = davies_bouldin_score(
            X_pca,
            labels
        )

        calinski_harabasz = calinski_harabasz_score(
            X_pca,
            labels
        )

        sse = kmeans.inertia_

        kmeans_results.append({
            "FeatureSet": feature_set_name,
            "K": k,
            "SSE": sse,
            "Silhouette": silhouette,
            "DaviesBouldin": davies_bouldin,
            "CalinskiHarabasz": calinski_harabasz,
            "MinClusterSize": min_cluster_size,
            "MinClusterPct": min_cluster_pct
        })

        print(
            f"K={k} | "
            f"Sil={silhouette:.4f} | "
            f"DB={davies_bouldin:.4f} | "
            f"CH={calinski_harabasz:.2f} | "
            f"MinCluster={min_cluster_pct:.2f}%"
        )

kmeans_results_df = pd.DataFrame(kmeans_results)

print("\n========================================")
print("K-MEANS SCREENING COMPLETE")
print("========================================")

display(
    kmeans_results_df.round(4)
)


Testing: RFM
K=3 | Sil=0.3755 | DB=0.9495 | CH=4282.34 | MinCluster=17.32%
K=4 | Sil=0.3820 | DB=0.8996 | CH=4114.00 | MinCluster=15.80%
K=5 | Sil=0.3661 | DB=0.8640 | CH=4086.59 | MinCluster=6.99%
K=6 | Sil=0.3701 | DB=0.8646 | CH=4108.35 | MinCluster=7.59%
K=7 | Sil=0.3721 | DB=0.8303 | CH=4111.29 | MinCluster=5.26%
K=8 | Sil=0.3703 | DB=0.8298 | CH=4063.14 | MinCluster=4.98%
K=9 | Sil=0.3666 | DB=0.8352 | CH=4014.07 | MinCluster=3.81%
K=10 | Sil=0.3561 | DB=0.8324 | CH=3957.50 | MinCluster=1.75%

Testing: RFM_Behavioral
K=3 | Sil=0.3827 | DB=0.9985 | CH=2866.05 | MinCluster=24.71%
K=4 | Sil=0.4135 | DB=0.8736 | CH=3235.92 | MinCluster=14.44%
K=5 | Sil=0.3807 | DB=0.9810 | CH=3042.02 | MinCluster=8.93%
K=6 | Sil=0.3724 | DB=1.0213 | CH=2927.57 | MinCluster=7.17%
K=7 | Sil=0.3619 | DB=1.0580 | CH=2732.11 | MinCluster=6.97%
K=8 | Sil=0.3473 | DB=1.0458 | CH=2601.96 | MinCluster=2.58%
K=9 | Sil=0.3447 | DB=1.0461 | CH=2507.08 | MinCluster=2.40%
K=10 | Sil=0.3337 | DB=1.0326 | CH=2443.5

,FeatureSet,K,SSE,Silhouette,DaviesBouldin,CalinskiHarabasz,MinClusterSize,MinClusterPct
0,RFM,3,4099.7357,0.3755,0.9495,4282.3365,751,17.3241
1,RFM,4,3170.4160,0.3820,0.8996,4114.0041,685,15.8016
2,RFM,5,2555.9573,0.3661,0.8640,4086.5914,303,6.9896
3,RFM,6,2124.4016,0.3701,0.8646,4108.3451,329,7.5894
4,RFM,7,1821.7795,0.3721,0.8303,4111.2912,228,5.2595
5,RFM,8,1611.6424,0.3703,0.8298,4063.1417,216,4.9827
6,RFM,9,1449.0070,0.3666,0.8352,4014.0746,165,3.8062
7,RFM,10,1321.5825,0.3561,0.8324,3957.5011,76,1.7532
8,RFM_Behavioral,3,8589.4244,0.3827,0.9985,2866.0523,1071,24.7059
9,RFM_Behavioral,4,6156.1696,0.4135,0.8736,3235.9235,626,14.4406


In [36]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 17: K-Means Stability Analysis
# ============================================================

stability_seeds = [42, 52, 62, 72, 82]

stability_results = []

X_temporal = pca_feature_sets["RFM_Temporal"]

for k in [3, 4, 5, 6]:

    for seed in stability_seeds:

        model = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=20,
            max_iter=300
        )

        labels = model.fit_predict(X_temporal)

        cluster_counts = np.bincount(labels)

        stability_results.append({
            "K": k,
            "Seed": seed,
            "SSE": model.inertia_,
            "Silhouette": silhouette_score(
                X_temporal,
                labels
            ),
            "DaviesBouldin": davies_bouldin_score(
                X_temporal,
                labels
            ),
            "CalinskiHarabasz": calinski_harabasz_score(
                X_temporal,
                labels
            ),
            "MinClusterSize": cluster_counts.min(),
            "MinClusterPct": (
                cluster_counts.min()
                / len(labels)
                * 100
            )
        })

stability_df = pd.DataFrame(stability_results)

print("RFM_Temporal stability results:")

display(
    stability_df.round(4)
)

# ------------------------------------------------------------
# Aggregate stability statistics
# ------------------------------------------------------------

stability_summary = (
    stability_df
    .groupby("K")
    .agg(
        SilhouetteMean=("Silhouette", "mean"),
        SilhouetteStd=("Silhouette", "std"),
        DBMean=("DaviesBouldin", "mean"),
        DBStd=("DaviesBouldin", "std"),
        CHMean=("CalinskiHarabasz", "mean"),
        CHStd=("CalinskiHarabasz", "std"),
        MinClusterPctMean=("MinClusterPct", "mean"),
        MinClusterPctMin=("MinClusterPct", "min")
    )
    .reset_index()
)

print("\nStability summary:")

display(
    stability_summary.round(4)
)

RFM_Temporal stability results:


,K,Seed,SSE,Silhouette,DaviesBouldin,CalinskiHarabasz,MinClusterSize,MinClusterPct
0,3,42,6418.6036,0.5874,0.6026,5813.9042,992,22.8835
1,3,52,6418.6036,0.5874,0.6026,5813.9042,992,22.8835
2,3,62,6418.6036,0.5874,0.6026,5813.9042,992,22.8835
3,3,72,6418.6036,0.5874,0.6026,5813.9042,992,22.8835
4,3,82,6418.6036,0.5874,0.6026,5813.9042,992,22.8835
5,4,42,4317.0762,0.5504,0.7538,6464.1634,689,15.8939
6,4,52,4317.0762,0.5504,0.7538,6464.1634,689,15.8939
7,4,62,4317.0762,0.5504,0.7538,6464.1634,689,15.8939
8,4,72,4317.0762,0.5504,0.7538,6464.1634,689,15.8939
9,4,82,4317.0762,0.5504,0.7538,6464.1634,689,15.8939



Stability summary:


,K,SilhouetteMean,SilhouetteStd,DBMean,DBStd,CHMean,CHStd,MinClusterPctMean,MinClusterPctMin
0,3,0.5874,0.0000,0.6026,0.0000,5813.9042,0.0000,22.8835,22.8835
1,4,0.5504,0.0000,0.7538,0.0000,6464.1634,0.0000,15.8939,15.8939
2,5,0.5037,0.0000,0.7804,0.0000,6142.5283,0.0006,10.3160,10.3114
3,6,0.4685,0.0009,0.8651,0.0032,6162.2260,1.2236,7.5479,6.8743


In [37]:
# ============================================================
# PHASE 2 — SEGMENTATION REFINEMENT
# Cell 18: Cluster Profile Analysis
# ============================================================

profile_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "TotalQuantity",
    "ProductDiversity",
    "AverageBasketQuantity",
    "AverageTransactionValue",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

profile_data = customer_features[
    profile_features
].copy()

profile_results = {}

for k in [3, 4, 5]:

    model = KMeans(
        n_clusters=k,
        random_state=82,
        n_init=20,
        max_iter=300
    )

    labels = model.fit_predict(
        pca_feature_sets["RFM_Temporal"]
    )

    profile_df = profile_data.copy()
    profile_df["Cluster"] = labels

    # Cluster sizes
    cluster_sizes = (
        profile_df["Cluster"]
        .value_counts()
        .sort_index()
        .rename("Customers")
        .to_frame()
    )

    cluster_sizes["Percentage"] = (
        cluster_sizes["Customers"]
        / len(profile_df)
        * 100
    )

    # Cluster means
    cluster_profile = (
        profile_df
        .groupby("Cluster")[profile_features]
        .mean()
    )

    profile_results[k] = {
        "labels": labels,
        "sizes": cluster_sizes,
        "profile": cluster_profile
    }

    print("\n" + "=" * 70)
    print(f"K = {k}")
    print("=" * 70)

    print("\nCluster sizes:")
    display(cluster_sizes.round(2))

    print("\nCluster profile:")
    display(cluster_profile.round(2))


K = 3

Cluster sizes:


,Customers,Percentage
Cluster,,
0,1968,45.40
1,992,22.88
2,1375,31.72



Cluster profile:


,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
Cluster,,,,,,,,,,,,,
0,40.87,7.33,3721.92,257.75,85.91,2158.93,96.19,266.04,456.15,35.57,0.56,25.78,0.0
1,255.42,1.56,536.75,26.92,55.27,333.19,24.42,251.47,387.53,-0.99,-0.98,-0.99,0.0
2,49.51,1.78,645.00,22.52,46.73,409.19,38.36,242.17,377.42,0.00,0.00,0.00,1.0



K = 4

Cluster sizes:


...

,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
Cluster,,,,,,,,,,,,,
0,257.94,1.51,584.33,24.42,55.11,353.34,24.26,261.22,408.01,-1.00,-1.00,-1.00,0.0
1,55.44,3.73,1196.73,227.45,111.83,748.55,63.99,208.29,333.68,1.04,0.43,1.96,0.0
2,49.54,1.77,643.44,22.43,46.75,408.58,38.23,242.28,377.54,0.00,0.00,0.00,1.0
3,15.72,14.03,8326.40,311.87,36.31,4737.85,155.30,360.49,655.88,99.63,0.77,69.95,0.0



K = 5

Cluster sizes:


,Customers,Percentage
Cluster,,
0,1299,29.97
1,447,10.31
2,973,22.45
3,688,15.87
4,928,21.41



Cluster profile:


,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
Cluster,,,,,,,,,,,,,
0,55.44,3.73,1196.73,227.45,111.83,748.55,63.99,208.29,333.68,1.04,0.43,1.96,0.0
1,22.19,3.06,1283.67,57.33,36.98,793.27,67.47,318.74,518.60,0.00,0.00,0.00,1.0
2,257.94,1.51,584.33,24.42,55.11,353.34,24.26,261.22,408.01,-1.00,-1.00,-1.00,0.0
3,15.74,14.04,8334.44,312.12,36.35,4742.92,155.22,360.87,656.52,99.77,0.77,70.05,0.0
4,62.67,1.16,337.37,5.75,51.42,224.20,24.33,205.29,309.41,0.00,0.00,0.00,1.0


In [38]:
# ============================================================
# PHASE 2 — CLUSTER PROFILE COMPARISON
# Cell 18B: Compact Profiles
# ============================================================

important_profile_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

for k in [3, 4, 5]:

    print("\n" + "=" * 70)
    print(f"K = {k}")
    print("=" * 70)

    sizes = profile_results[k]["sizes"].copy()
    profile = profile_results[k]["profile"][
        important_profile_features
    ].copy()

    print("\nCluster sizes:")
    print(sizes.round(2).to_string())

    print("\nCluster profiles:")
    print(profile.round(2).to_string())


K = 3

Cluster sizes:
         Customers  Percentage
Cluster                       
0             1968       45.40
1              992       22.88
2             1375       31.72

Cluster profiles:
         Recency  Frequency  Monetary  CustomerLifespanDays  AveragePurchaseIntervalDays  SpendingTrend  FrequencyTrend  QuantityTrend  NewCustomer
Cluster                                                                                                                                            
0          40.87       7.33   3721.92                257.75                        85.91          35.57            0.56          25.78          0.0
1         255.42       1.56    536.75                 26.92                        55.27          -0.99           -0.98          -0.99          0.0
2          49.51       1.78    645.00                 22.52                        46.73           0.00            0.00           0.00          1.0

K = 4

Cluster sizes:
         Customers  Percentage
Cluster  

In [39]:
# ============================================================
# PHASE 2 — CLUSTER PROFILE COMPARISON
# Cell 18C: Compact K=3/4/5 Comparison
# ============================================================

key_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

comparison_rows = []

for k in [3, 4, 5]:

    sizes = profile_results[k]["sizes"]
    profile = profile_results[k]["profile"]

    for cluster in profile.index:

        row = {
            "K": k,
            "Cluster": cluster,
            "Customers": sizes.loc[cluster, "Customers"],
            "Percentage": sizes.loc[cluster, "Percentage"]
        }

        for feature in key_features:
            row[feature] = profile.loc[cluster, feature]

        comparison_rows.append(row)

profile_comparison = pd.DataFrame(comparison_rows)

print("Complete compact cluster comparison:")
display(
    profile_comparison.round(2)
)

Complete compact cluster comparison:


,K,Cluster,Customers,Percentage,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
0,3,0,1968,45.40,40.87,7.33,3721.92,257.75,85.91,35.57,0.56,25.78,0.0
1,3,1,992,22.88,255.42,1.56,536.75,26.92,55.27,-0.99,-0.98,-0.99,0.0
2,3,2,1375,31.72,49.51,1.78,645.00,22.52,46.73,0.00,0.00,0.00,1.0
3,4,0,973,22.45,257.94,1.51,584.33,24.42,55.11,-1.00,-1.00,-1.00,0.0
4,4,1,1299,29.97,55.44,3.73,1196.73,227.45,111.83,1.04,0.43,1.96,0.0
5,4,2,1374,31.70,49.54,1.77,643.44,22.43,46.75,0.00,0.00,0.00,1.0
6,4,3,689,15.89,15.72,14.03,8326.40,311.87,36.31,99.63,0.77,69.95,0.0
7,5,0,1299,29.97,55.44,3.73,1196.73,227.45,111.83,1.04,0.43,1.96,0.0
8,5,1,447,10.31,22.19,3.06,1283.67,57.33,36.98,0.00,0.00,0.00,1.0
9,5,2,973,22.45,257.94,1.51,584.33,24.42,55.11,-1.00,-1.00,-1.00,0.0


In [40]:
print(
    profile_comparison[
        profile_comparison["K"] == 5
    ].round(2).to_string(index=False)
)

 K  Cluster  Customers  Percentage  Recency  Frequency  Monetary  CustomerLifespanDays  AveragePurchaseIntervalDays  SpendingTrend  FrequencyTrend  QuantityTrend  NewCustomer
 5        0       1299       29.97    55.44       3.73   1196.73                227.45                       111.83           1.04            0.43           1.96          0.0
 5        1        447       10.31    22.19       3.06   1283.67                 57.33                        36.98           0.00            0.00           0.00          1.0
 5        2        973       22.45   257.94       1.51    584.33                 24.42                        55.11          -1.00           -1.00          -1.00          0.0
 5        3        688       15.87    15.74      14.04   8334.44                312.12                        36.35          99.77            0.77          70.05          0.0
 5        4        928       21.41    62.67       1.16    337.37                  5.75                        51.42          

In [41]:
# ============================================================
# PHASE 2 — QLDE EXPERIMENT SETUP
# Cell 19
# ============================================================

qlde_feature_set = pca_feature_sets["RFM_Temporal"]

candidate_k_values = [3, 4, 5]
seeds = [42, 52, 62, 72, 82]

print("QLDE experiment configuration")
print("-" * 50)
print("Feature set : RFM_Temporal")
print("Features    : Recency, Frequency, Monetary,")
print("              RecentSpendingShare, ActivityConsistency, NewCustomer")
print("PCA         : 90% variance")
print("K values    :", candidate_k_values)
print("Seeds       :", seeds)
print("Customers   :", qlde_feature_set.shape[0])
print("Dimensions  :", qlde_feature_set.shape[1])

QLDE experiment configuration
--------------------------------------------------
Feature set : RFM_Temporal
Features    : Recency, Frequency, Monetary,
              RecentSpendingShare, ActivityConsistency, NewCustomer
PCA         : 90% variance
K values    : [3, 4, 5]
Seeds       : [42, 52, 62, 72, 82]
Customers   : 4335
Dimensions  : 3


In [43]:
# ============================================================
# Cell 20 — Inspect existing segmentation pipeline
# ============================================================

import inspect
import segmentation_pipeline

print("segmentation_pipeline location:")
print(segmentation_pipeline.__file__)

print("\nAvailable names:")
print([
    name for name in dir(segmentation_pipeline)
    if not name.startswith("_")
])

segmentation_pipeline location:
c:\Users\akshi\Desktop\Behavioral-customer-segmentation\src\segmentation_pipeline.py

Available names:
['ExperimentConfig', 'FEATURE_COLUMNS', 'KMeans', 'PCA', 'Path', 'annotations', 'asdict', 'assign_and_valid_sse', 'calinski_harabasz_score', 'centroids_to_chromosome', 'chromosome_to_centroids', 'dataclass', 'davies_bouldin_score', 'dump', 'evaluate_centroid_model', 'initialize_population', 'json', 'kmeans_plusplus', 'main', 'np', 'pd', 'run_experiment', 'run_qlde', 'silhouette_score']


In [44]:
# ============================================================
# PHASE 2 — LOAD EXISTING PHASE 1 QLDE
# Cell 20
# ============================================================

from segmentation_pipeline import (
    run_qlde,
    evaluate_centroid_model,
    assign_and_valid_sse
)

print("Phase 1 QLDE functions loaded successfully.")

Phase 1 QLDE functions loaded successfully.


In [46]:
import inspect

print(inspect.signature(run_qlde))
print("\nFull function:")
print(inspect.getsource(run_qlde))

(x: 'np.ndarray', k: 'int', seed: 'int', config: 'ExperimentConfig') -> 'dict'

Full function:
def run_qlde(x: np.ndarray, k: int, seed: int, config: ExperimentConfig) -> dict:
    """QLDE with valid fitness, individual Q tables, adaptive F, and elitist diversity resets."""
    rng = np.random.default_rng(seed)
    _, d = x.shape
    dimension = k * d
    pop_size = min(config.population_cap, max(30, 5 * dimension))
    population, lower, upper = initialize_population(x, k, pop_size, rng)
    fitness = np.array([assign_and_valid_sse(x, chromosome_to_centroids(c, k, d))[0] for c in population])
    actions = np.asarray(config.q_actions)
    # Individual policies avoid order-dependent sharing of unrelated trajectories.
    q_tables = np.zeros((pop_size, 3, len(actions)))
    states = np.ones(pop_size, dtype=int)
    f_values = np.full(pop_size, config.f_initial)
    stagnation = np.zeros(pop_size, dtype=int)
    best_idx = int(fitness.argmin())
    best = population[best_idx].copy()
    

In [47]:
# ============================================================
# PHASE 2 — QLDE CONFIGURATION
# Cell 21B
# ============================================================

from segmentation_pipeline import ExperimentConfig

config = ExperimentConfig()

print("QLDE configuration loaded.")
print(config)

QLDE configuration loaded.
ExperimentConfig(pca_variance_target=0.9, candidate_pcs=(4, 5, 7), k_values=(2, 3, 4, 5, 6, 7, 8, 9, 10), seeds=(42, 52, 62, 72, 82), population_cap=80, iterations=100, f_initial=0.55, f_min=0.1, f_max=0.9, crossover_rate=0.9, q_alpha=0.15, q_gamma=0.9, q_epsilon=0.2, q_actions=(-0.05, -0.02, 0.0, 0.02, 0.05), stagnation_limit=15, reinitialization_fraction=0.1, kmeans_max_iter=300, kmeans_n_init=20)


In [48]:
# ============================================================
# PHASE 2 — QLDE TEST RUN
# Cell 21C
# ============================================================

test_k = 3
test_seed = 82

qlde_test = run_qlde(
    qlde_feature_set,
    test_k,
    test_seed,
    config
)

print("QLDE test completed.")
print("Returned type:", type(qlde_test))
print("Returned keys:", qlde_test.keys())

QLDE test completed.
Returned type: <class 'dict'>
Returned keys: dict_keys(['centroids', 'history', 'population_size'])


In [49]:
# ============================================================
# PHASE 2 — EVALUATE QLDE TEST
# Cell 22
# ============================================================

qlde_centroids = qlde_test["centroids"]

qlde_test_metrics = evaluate_centroid_model(
    qlde_feature_set,
    qlde_centroids
)

print("QLDE test metrics:")
print(qlde_test_metrics)

QLDE test metrics:
{'SSE': 6444.214326518322, 'EmptyClusters': 0, 'MinClusterSize': 990, 'MaxClusterSize': 1970, 'labels': array([1, 0, 0, ..., 2, 0, 0], shape=(4335,)), 'counts': array([1970,  990, 1375]), 'Silhouette': 0.5875478806917253, 'DaviesBouldin': 0.6023338719907193, 'CalinskiHarabasz': 5813.702946642133}


In [50]:
# ============================================================
# PHASE 2 — FULL QLDE EXPERIMENT
# Cell 23
# ============================================================

qlde_results = []

for k in candidate_k_values:

    print(f"\nRunning QLDE for K={k}...")

    for seed in seeds:

        result = run_qlde(
            qlde_feature_set,
            k,
            seed,
            config
        )

        metrics = evaluate_centroid_model(
            qlde_feature_set,
            result["centroids"]
        )

        qlde_results.append({
            "K": k,
            "Seed": seed,
            "SSE": metrics["SSE"],
            "Silhouette": metrics["Silhouette"],
            "DaviesBouldin": metrics["DaviesBouldin"],
            "CalinskiHarabasz": metrics["CalinskiHarabasz"],
            "EmptyClusters": metrics["EmptyClusters"],
            "MinClusterSize": metrics["MinClusterSize"],
            "MaxClusterSize": metrics["MaxClusterSize"]
        })

        print(
            f"  Seed {seed}: "
            f"Sil={metrics['Silhouette']:.4f}, "
            f"DB={metrics['DaviesBouldin']:.4f}, "
            f"CH={metrics['CalinskiHarabasz']:.2f}"
        )

qlde_results_df = pd.DataFrame(qlde_results)

print("\n" + "="*70)
print("QLDE experiment completed.")
print("Total runs:", len(qlde_results_df))


Running QLDE for K=3...
  Seed 42: Sil=0.5870, DB=0.6035, CH=5812.10
  Seed 52: Sil=0.5867, DB=0.6035, CH=5813.08
  Seed 62: Sil=0.5870, DB=0.6030, CH=5813.77
  Seed 72: Sil=0.5876, DB=0.6020, CH=5812.95
  Seed 82: Sil=0.5875, DB=0.6023, CH=5813.70

Running QLDE for K=4...
  Seed 42: Sil=0.5527, DB=0.7470, CH=6447.90
  Seed 52: Sil=0.5508, DB=0.7520, CH=6457.92
  Seed 62: Sil=0.5515, DB=0.7495, CH=6451.36
  Seed 72: Sil=0.5460, DB=0.7626, CH=6445.61
  Seed 82: Sil=0.5436, DB=0.7684, CH=6416.03

Running QLDE for K=5...
  Seed 42: Sil=0.5056, DB=0.7726, CH=6097.29
  Seed 52: Sil=0.5033, DB=0.7766, CH=6118.93
  Seed 62: Sil=0.4891, DB=0.8050, CH=6005.35
  Seed 72: Sil=0.5106, DB=0.8374, CH=5822.38
  Seed 82: Sil=0.4930, DB=0.7978, CH=6081.44

QLDE experiment completed.
Total runs: 15


In [51]:
# ============================================================
# PHASE 2 — QLDE + K-MEANS
# Cell 24
# ============================================================

hybrid_results = []

for k in candidate_k_values:

    print(f"\nRunning QLDE + K-Means for K={k}...")

    for seed in seeds:

        # Step 1: QLDE
        qlde_result = run_qlde(
            qlde_feature_set,
            k,
            seed,
            config
        )

        qlde_centroids = qlde_result["centroids"]

        # Step 2: K-Means refinement
        kmeans_model = KMeans(
            n_clusters=k,
            init=qlde_centroids,
            n_init=1,
            max_iter=config.kmeans_max_iter,
            random_state=seed
        )

        labels = kmeans_model.fit_predict(qlde_feature_set)

        # Step 3: Evaluate final hybrid solution
        sse = float(kmeans_model.inertia_)
        silhouette = silhouette_score(qlde_feature_set, labels)
        db = davies_bouldin_score(qlde_feature_set, labels)
        ch = calinski_harabasz_score(qlde_feature_set, labels)

        counts = np.bincount(labels, minlength=k)

        hybrid_results.append({
            "K": k,
            "Seed": seed,
            "SSE": sse,
            "Silhouette": silhouette,
            "DaviesBouldin": db,
            "CalinskiHarabasz": ch,
            "EmptyClusters": int(np.sum(counts == 0)),
            "MinClusterSize": int(counts.min()),
            "MaxClusterSize": int(counts.max())
        })

        print(
            f"  Seed {seed}: "
            f"Sil={silhouette:.4f}, "
            f"DB={db:.4f}, "
            f"CH={ch:.2f}"
        )

hybrid_results_df = pd.DataFrame(hybrid_results)

print("\n" + "="*70)
print("QLDE + K-Means experiment completed.")
print("Total runs:", len(hybrid_results_df))


Running QLDE + K-Means for K=3...
  Seed 42: Sil=0.5873, DB=0.6027, CH=5813.90
  Seed 52: Sil=0.5873, DB=0.6027, CH=5813.90
  Seed 62: Sil=0.5873, DB=0.6027, CH=5813.90
  Seed 72: Sil=0.5874, DB=0.6026, CH=5813.90
  Seed 82: Sil=0.5874, DB=0.6026, CH=5813.90

Running QLDE + K-Means for K=4...
  Seed 42: Sil=0.5504, DB=0.7538, CH=6464.16
  Seed 52: Sil=0.5504, DB=0.7538, CH=6464.16
  Seed 62: Sil=0.5504, DB=0.7538, CH=6464.16
  Seed 72: Sil=0.5500, DB=0.7547, CH=6464.15
  Seed 82: Sil=0.5500, DB=0.7548, CH=6464.14

Running QLDE + K-Means for K=5...
  Seed 42: Sil=0.5037, DB=0.7804, CH=6142.53
  Seed 52: Sil=0.5036, DB=0.7805, CH=6142.53
  Seed 62: Sil=0.5033, DB=0.7815, CH=6142.47
  Seed 72: Sil=0.5111, DB=0.8540, CH=5854.98
  Seed 82: Sil=0.5035, DB=0.7808, CH=6142.50

QLDE + K-Means experiment completed.
Total runs: 15


In [52]:
# ============================================================
# PHASE 2 — MODEL COMPARISON
# Cell 25
# ============================================================

# Standard K-Means results from Cell 16
# Rename/copy into a consistent format if necessary.

standard_results_df = standard_results.copy()

standard_results_df["Method"] = "Standard K-Means"
qlde_results_df["Method"] = "QLDE"
hybrid_results_df["Method"] = "QLDE + K-Means"

all_results = pd.concat(
    [
        standard_results_df,
        qlde_results_df,
        hybrid_results_df
    ],
    ignore_index=True
)

summary = (
    all_results
    .groupby(["Method", "K"])
    .agg(
        SSE_mean=("SSE", "mean"),
        SSE_std=("SSE", "std"),
        Silhouette_mean=("Silhouette", "mean"),
        Silhouette_std=("Silhouette", "std"),
        DB_mean=("DaviesBouldin", "mean"),
        DB_std=("DaviesBouldin", "std"),
        CH_mean=("CalinskiHarabasz", "mean"),
        CH_std=("CalinskiHarabasz", "std"),
        MinCluster_mean=("MinClusterSize", "mean"),
        MinCluster_min=("MinClusterSize", "min"),
        EmptyClusters_max=("EmptyClusters", "max")
    )
    .reset_index()
)

print("PHASE 2 MODEL COMPARISON")
print("=" * 100)

display(
    summary.round(4)
)

NameError: name 'standard_results' is not defined

In [53]:
# ============================================================
# PHASE 2 — FIND STANDARD K-MEANS RESULTS
# Cell 25A
# ============================================================

print("Variables containing 'result' or 'screen':")

for name in dir():
    if "result" in name.lower() or "screen" in name.lower():
        obj = globals()[name]
        if isinstance(obj, (pd.DataFrame, list, dict)):
            print(
                f"{name}: "
                f"{type(obj).__name__}, "
                f"shape/length = "
                f"{getattr(obj, 'shape', len(obj) if hasattr(obj, '__len__') else 'N/A')}"
            )

Variables containing 'result' or 'screen':
hybrid_results: list, shape/length = 15
hybrid_results_df: DataFrame, shape/length = (15, 9)
kmeans_results: list, shape/length = 40
kmeans_results_df: DataFrame, shape/length = (40, 8)
pca_results: list, shape/length = 5
pca_results_df: DataFrame, shape/length = (5, 8)
profile_results: dict, shape/length = 3
qlde_result: dict, shape/length = 3
qlde_results: list, shape/length = 15
qlde_results_df: DataFrame, shape/length = (15, 9)
result: dict, shape/length = 3
stability_results: list, shape/length = 20


In [54]:
# ============================================================
# PHASE 2 — FINAL MODEL COMPARISON
# Cell 25B
# ============================================================

# Standard K-Means results for RFM_Temporal, K=3,4,5
standard_results_df = kmeans_results_df[
    (kmeans_results_df["FeatureSet"] == "RFM_Temporal") &
    (kmeans_results_df["K"].isin(candidate_k_values))
].copy()

standard_results_df["Method"] = "Standard K-Means"

qlde_results_df["Method"] = "QLDE"
hybrid_results_df["Method"] = "QLDE + K-Means"

all_results = pd.concat(
    [
        standard_results_df,
        qlde_results_df,
        hybrid_results_df
    ],
    ignore_index=True
)

summary = (
    all_results
    .groupby(["Method", "K"])
    .agg(
        SSE_mean=("SSE", "mean"),
        SSE_std=("SSE", "std"),
        Silhouette_mean=("Silhouette", "mean"),
        Silhouette_std=("Silhouette", "std"),
        DB_mean=("DaviesBouldin", "mean"),
        DB_std=("DaviesBouldin", "std"),
        CH_mean=("CalinskiHarabasz", "mean"),
        CH_std=("CalinskiHarabasz", "std"),
        MinCluster_mean=("MinClusterSize", "mean"),
        MinCluster_min=("MinClusterSize", "min"),
        EmptyClusters_max=("EmptyClusters", "max")
    )
    .reset_index()
)

print("PHASE 2 — STANDARD K-MEANS vs QLDE vs QLDE + K-MEANS")
print("=" * 100)

display(summary.round(4))

PHASE 2 — STANDARD K-MEANS vs QLDE vs QLDE + K-MEANS


,Method,K,SSE_mean,SSE_std,Silhouette_mean,Silhouette_std,DB_mean,DB_std,CH_mean,CH_std,MinCluster_mean,MinCluster_min,EmptyClusters_max
0,QLDE,3,6434.1598,16.7116,0.5872,0.0004,0.6029,0.0007,5813.1208,0.6758,994.6,987,0.0
1,QLDE,4,4417.3093,68.2007,0.5489,0.0039,0.7559,0.0092,6443.7643,16.1840,706.0,626,0.0
2,QLDE,5,3733.2211,93.4988,0.5004,0.0090,0.7979,0.0260,6025.0782,121.1149,474.2,410,0.0
3,QLDE + K-Means,3,6418.6133,0.0132,0.5873,0.0001,0.6026,0.0001,5813.9009,0.0030,992.6,992,0.0
4,QLDE + K-Means,4,4317.0809,0.0070,0.5502,0.0002,0.7542,0.0005,6464.1548,0.0126,692.4,689,0.0
5,QLDE + K-Means,5,3572.3600,65.6720,0.5050,0.0034,0.7955,0.0327,6085.0015,128.5880,441.6,417,0.0
6,Standard K-Means,3,6418.6036,NaN,0.5874,NaN,0.6026,NaN,5813.9042,NaN,992.0,992,NaN
7,Standard K-Means,4,4317.0762,NaN,0.5504,NaN,0.7538,NaN,6464.1634,NaN,689.0,689,NaN
8,Standard K-Means,5,3542.9903,NaN,0.5037,NaN,0.7804,NaN,6142.5286,NaN,447.0,447,NaN


In [55]:
# ============================================================
# PHASE 2 — FINAL CANDIDATE PROFILES
# Cell 26
# ============================================================

final_profile_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "TotalQuantity",
    "ProductDiversity",
    "AverageBasketQuantity",
    "AverageTransactionValue",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

final_profile_data = customer_features[final_profile_features].copy()

final_candidate_profiles = {}

for k in [3, 4]:

    model = KMeans(
        n_clusters=k,
        random_state=82,
        n_init=20,
        max_iter=300
    )

    labels = model.fit_predict(qlde_feature_set)

    profile = final_profile_data.copy()
    profile["Cluster"] = labels

    cluster_profile = (
        profile
        .groupby("Cluster")[final_profile_features]
        .mean()
        .round(2)
    )

    cluster_sizes = (
        profile["Cluster"]
        .value_counts()
        .sort_index()
        .rename("Customers")
        .to_frame()
    )

    cluster_sizes["Percentage"] = (
        cluster_sizes["Customers"] / len(profile) * 100
    ).round(2)

    final_candidate_profiles[k] = {
        "model": model,
        "labels": labels,
        "profile": cluster_profile,
        "sizes": cluster_sizes
    }

    print("=" * 80)
    print(f"K = {k}")
    print("\nCluster sizes:")
    display(cluster_sizes)

    print("\nCluster profiles:")
    display(cluster_profile)

K = 3

Cluster sizes:


,Customers,Percentage
Cluster,,
0,1968,45.40
1,992,22.88
2,1375,31.72



Cluster profiles:


,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
Cluster,,,,,,,,,,,,,
0,40.87,7.33,3721.92,257.75,85.91,2158.93,96.19,266.04,456.15,35.57,0.56,25.78,0.0
1,255.42,1.56,536.75,26.92,55.27,333.19,24.42,251.47,387.53,-0.99,-0.98,-0.99,0.0
2,49.51,1.78,645.00,22.52,46.73,409.19,38.36,242.17,377.42,0.00,0.00,0.00,1.0


K = 4

Cluster sizes:


,Customers,Percentage
Cluster,,
0,973,22.45
1,1299,29.97
2,1374,31.70
3,689,15.89



Cluster profiles:


,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
Cluster,,,,,,,,,,,,,
0,257.94,1.51,584.33,24.42,55.11,353.34,24.26,261.22,408.01,-1.00,-1.00,-1.00,0.0
1,55.44,3.73,1196.73,227.45,111.83,748.55,63.99,208.29,333.68,1.04,0.43,1.96,0.0
2,49.54,1.77,643.44,22.43,46.75,408.58,38.23,242.28,377.54,0.00,0.00,0.00,1.0
3,15.72,14.03,8326.40,311.87,36.31,4737.85,155.30,360.49,655.88,99.63,0.77,69.95,0.0


In [56]:
# ============================================================
# Cell 26B — Complete K=4 profile
# ============================================================

print(
    final_candidate_profiles[4]["profile"]
    .loc[3]
    .round(2)
    .to_string()
)

Recency                          15.72
Frequency                        14.03
Monetary                       8326.40
CustomerLifespanDays            311.87
AveragePurchaseIntervalDays      36.31
TotalQuantity                  4737.85
ProductDiversity                155.30
AverageBasketQuantity           360.49
AverageTransactionValue         655.88
SpendingTrend                    99.63
FrequencyTrend                    0.77
QuantityTrend                    69.95
NewCustomer                       0.00


In [57]:
# ============================================================
# PHASE 2 — FINAL SEGMENT DEFINITIONS
# Cell 27
# ============================================================

segment_definitions = pd.DataFrame([
    {
        "Cluster": 0,
        "Status": "Inactive / Declining",
        "Evidence": "High recency, low frequency and monetary value, strongly negative trends",
        "MarketingObjective": "Reactivation"
    },
    {
        "Cluster": 1,
        "Status": "Established Active",
        "Evidence": "Established customers with moderate frequency/value and positive behavioral trends",
        "MarketingObjective": "Retention and relationship growth"
    },
    {
        "Cluster": 2,
        "Status": "New / Recently Acquired",
        "Evidence": "NewCustomer=1, short lifespan, low-to-moderate initial frequency/value",
        "MarketingObjective": "Onboarding and second-purchase conversion"
    },
    {
        "Cluster": 3,
        "Status": "Very High-Value Active",
        "Evidence": "Very high frequency, monetary value, product diversity and recent activity",
        "MarketingObjective": "Retention and high-value relationship development"
    }
])

display(segment_definitions)

,Cluster,Status,Evidence,MarketingObjective
0,0,Inactive / Declining,"High recency, low frequency and monetary value...",Reactivation
1,1,Established Active,Established customers with moderate frequency/...,Retention and relationship growth
2,2,New / Recently Acquired,"NewCustomer=1, short lifespan, low-to-moderate...",Onboarding and second-purchase conversion
3,3,Very High-Value Active,"Very high frequency, monetary value, product d...",Retention and high-value relationship development


In [58]:
# ============================================================
# PHASE 2 — PERSONALIZED MARKETING STRATEGY
# Cell 28
# ============================================================

marketing_strategy = pd.DataFrame([
    {
        "Cluster": 0,
        "Status": "Inactive / Declining",
        "CustomerBehavior": "Long time since purchase, low activity and strongly negative recent trends",
        "MarketingObjective": "Reactivation",
        "RecommendedAction": "Win-back campaign with personalized offers based on previous purchases",
        "PrimaryKPI": "Reactivation / repeat-purchase rate"
    },
    {
        "Cluster": 1,
        "Status": "Established Active",
        "CustomerBehavior": "Established customers with moderate purchasing activity and positive trends",
        "MarketingObjective": "Retention and relationship growth",
        "RecommendedAction": "Cross-sell, product recommendations and loyalty-oriented offers",
        "PrimaryKPI": "Repeat-purchase rate and customer value"
    },
    {
        "Cluster": 2,
        "Status": "New / Recently Acquired",
        "CustomerBehavior": "Recently acquired customers with short customer lifespan and limited purchase history",
        "MarketingObjective": "Onboarding and second-purchase conversion",
        "RecommendedAction": "Welcome journey, product discovery recommendations and second-purchase incentive",
        "PrimaryKPI": "Second-purchase conversion rate"
    },
    {
        "Cluster": 3,
        "Status": "Very High-Value Active",
        "CustomerBehavior": "Very high purchase frequency, monetary value, product diversity and recent activity",
        "MarketingObjective": "Retention and high-value relationship development",
        "RecommendedAction": "Personalized premium recommendations, loyalty benefits and early-access offers",
        "PrimaryKPI": "Retention rate and customer value"
    }
])

display(marketing_strategy)

,Cluster,Status,CustomerBehavior,MarketingObjective,RecommendedAction,PrimaryKPI
0,0,Inactive / Declining,"Long time since purchase, low activity and str...",Reactivation,Win-back campaign with personalized offers bas...,Reactivation / repeat-purchase rate
1,1,Established Active,Established customers with moderate purchasing...,Retention and relationship growth,"Cross-sell, product recommendations and loyalt...",Repeat-purchase rate and customer value
2,2,New / Recently Acquired,Recently acquired customers with short custome...,Onboarding and second-purchase conversion,"Welcome journey, product discovery recommendat...",Second-purchase conversion rate
3,3,Very High-Value Active,"Very high purchase frequency, monetary value, ...",Retention and high-value relationship development,"Personalized premium recommendations, loyalty ...",Retention rate and customer value


In [59]:
# ============================================================
# PHASE 2 — FINAL CUSTOMER SEGMENTATION OUTPUT
# Cell 29
# ============================================================

# Use the validated K=4 Standard K-Means model
final_model = final_candidate_profiles[4]["model"]
final_labels = final_candidate_profiles[4]["labels"]

# Start with CustomerID
final_customer_segments = customer_features[["CustomerID"]].copy()

# Add cluster
final_customer_segments["Cluster"] = final_labels

# Add segment information
segment_mapping = segment_definitions[
    ["Cluster", "Status", "MarketingObjective"]
].copy()

final_customer_segments = final_customer_segments.merge(
    segment_mapping,
    on="Cluster",
    how="left"
)

# Add key behavioral features
behavioral_output_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

final_customer_segments = final_customer_segments.merge(
    customer_features[
        ["CustomerID"] + behavioral_output_features
    ],
    on="CustomerID",
    how="left"
)

print("Final Phase 2 customer segmentation:")
print("Shape:", final_customer_segments.shape)

display(final_customer_segments.head(10))

Final Phase 2 customer segmentation:
Shape: (4335, 13)


,CustomerID,Cluster,Status,MarketingObjective,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
0,12346.0,0,Inactive / Declining,Reactivation,326,1,77183.60,0,54.025694,-1.000000,-1.000000,-1.000000,0
1,12347.0,3,Very High-Value Active,Retention and high-value relationship development,2,7,4310.00,365,60.839699,0.112775,-0.125000,0.017390,0
2,12348.0,1,Established Active,Retention and relationship growth,75,4,1437.24,282,94.250926,-0.730133,-0.611111,-0.880907,0
3,12349.0,2,New / Recently Acquired,Onboarding and second-purchase conversion,19,1,1457.55,0,54.025694,0.000000,0.000000,0.000000,1
4,12350.0,0,Inactive / Declining,Reactivation,310,1,294.40,0,54.025694,-1.000000,-1.000000,-1.000000,0
5,12352.0,1,Established Active,Retention and relationship growth,36,7,1385.74,260,43.347685,0.353476,-0.125000,0.297858,0
6,12353.0,0,Inactive / Declining,Reactivation,204,1,89.00,0,54.025694,-1.000000,-1.000000,-1.000000,0
7,12354.0,0,Inactive / Declining,Reactivation,232,1,1079.40,0,54.025694,-1.000000,-1.000000,-1.000000,0
8,12355.0,0,Inactive / Declining,Reactivation,214,1,459.40,0,54.025694,-1.000000,-1.000000,-1.000000,0
9,12356.0,1,Established Active,Retention and relationship growth,23,3,2487.43,302,151.475694,-0.971975,-0.416667,-0.996280,0


In [60]:
# ============================================================
# PHASE 2 — FINAL VALIDATION
# Cell 30
# ============================================================

print("=" * 70)
print("FINAL PHASE 2 VALIDATION")
print("=" * 70)

# 1. Row count
print("\n1. Customer count:")
print("Expected:", len(customer_features))
print("Actual  :", len(final_customer_segments))

# 2. Duplicate customers
print("\n2. Duplicate CustomerIDs:")
print(final_customer_segments["CustomerID"].duplicated().sum())

# 3. Missing values
print("\n3. Missing values:")
missing = final_customer_segments.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

# 4. Cluster distribution
print("\n4. Cluster distribution:")
print(
    final_customer_segments["Cluster"]
    .value_counts()
    .sort_index()
)

# 5. Segment distribution
print("\n5. Segment distribution:")
print(
    final_customer_segments["Status"]
    .value_counts()
)

# 6. Check all customers assigned
print("\n6. Unassigned customers:")
print(final_customer_segments["Cluster"].isna().sum())

# 7. Check cluster IDs
print("\n7. Cluster IDs:")
print(sorted(final_customer_segments["Cluster"].unique()))

print("\nValidation complete.")

FINAL PHASE 2 VALIDATION

1. Customer count:
Expected: 4335
Actual  : 4335

2. Duplicate CustomerIDs:
0

3. Missing values:
No missing values

4. Cluster distribution:
Cluster
0     973
1    1299
2    1374
3     689
Name: count, dtype: int64

5. Segment distribution:
Status
New / Recently Acquired    1374
Established Active         1299
Inactive / Declining        973
Very High-Value Active      689
Name: count, dtype: int64

6. Unassigned customers:
0

7. Cluster IDs:
[np.int32(0), np.int32(1), np.int32(2), np.int32(3)]

Validation complete.


In [61]:
# ============================================================
# PHASE 2 — SAVE FINAL OUTPUT
# Cell 31
# ============================================================

PHASE2_RESULTS_DIR = PROJECT_ROOT / "results" / "phase2"
PHASE2_TABLES_DIR = PHASE2_RESULTS_DIR / "tables"

PHASE2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PHASE2_TABLES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Final customer segmentation
final_customer_segments.to_csv(
    PHASE2_RESULTS_DIR / "phase2_customer_segments.csv",
    index=False
)

# 2. Segment definitions
segment_definitions.to_csv(
    PHASE2_TABLES_DIR / "phase2_segment_definitions.csv",
    index=False
)

# 3. Marketing strategy
marketing_strategy.to_csv(
    PHASE2_TABLES_DIR / "phase2_marketing_strategy.csv",
    index=False
)

# 4. Model comparison
summary.to_csv(
    PHASE2_TABLES_DIR / "phase2_model_comparison.csv",
    index=False
)

# 5. QLDE experiment results
qlde_results_df.to_csv(
    PHASE2_TABLES_DIR / "phase2_qlde_results.csv",
    index=False
)

# 6. QLDE + K-Means results
hybrid_results_df.to_csv(
    PHASE2_TABLES_DIR / "phase2_qlde_kmeans_results.csv",
    index=False
)

print("Phase 2 outputs saved successfully.")
print("\nFiles:")

for file in sorted(PHASE2_RESULTS_DIR.rglob("*.csv")):
    print(" -", file.relative_to(PROJECT_ROOT))

Phase 2 outputs saved successfully.

Files:
 - results\phase2\phase2_customer_segments.csv
 - results\phase2\tables\phase2_marketing_strategy.csv
 - results\phase2\tables\phase2_model_comparison.csv
 - results\phase2\tables\phase2_qlde_kmeans_results.csv
 - results\phase2\tables\phase2_qlde_results.csv
 - results\phase2\tables\phase2_segment_definitions.csv


In [63]:
# ============================================================
# PHASE 2 — PERSONALIZED RECOMMENDATION ENGINE
# Cell 32
# ============================================================

def generate_recommendation(row):

    if row["Cluster"] == 0:
        return {
            "Recommendation": (
                "Win-back campaign with personalized offers "
                "based on previous purchases"
            ),
            "CampaignType": "Reactivation",
            "Priority": "High"
        }

    elif row["Cluster"] == 1:
        return {
            "Recommendation": (
                "Cross-sell complementary products and provide "
                "loyalty-oriented offers"
            ),
            "CampaignType": "Retention / Cross-sell",
            "Priority": "Medium"
        }

    elif row["Cluster"] == 2:
        return {
            "Recommendation": (
                "Welcome journey with product discovery "
                "and second-purchase incentive"
            ),
            "CampaignType": "New Customer Onboarding",
            "Priority": "High"
        }

    elif row["Cluster"] == 3:
        return {
            "Recommendation": (
                "Premium personalized recommendations, loyalty "
                "benefits and early-access offers"
            ),
            "CampaignType": "High-Value Retention",
            "Priority": "High"
        }


recommendations = final_customer_segments.apply(
    generate_recommendation,
    axis=1,
    result_type="expand"
)

final_customer_segments = pd.concat(
    [final_customer_segments, recommendations],
    axis=1
)

print("Recommendation engine created.")
print("Final dataset shape:", final_customer_segments.shape)

display(final_customer_segments.head(10))

Recommendation engine created.
Final dataset shape: (4335, 16)


,CustomerID,Cluster,Status,MarketingObjective,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer,Recommendation,CampaignType,Priority
0,12346.0,0,Inactive / Declining,Reactivation,326,1,77183.60,0,54.025694,-1.000000,-1.000000,-1.000000,0,Win-back campaign with personalized offers bas...,Reactivation,High
1,12347.0,3,Very High-Value Active,Retention and high-value relationship development,2,7,4310.00,365,60.839699,0.112775,-0.125000,0.017390,0,"Premium personalized recommendations, loyalty ...",High-Value Retention,High
2,12348.0,1,Established Active,Retention and relationship growth,75,4,1437.24,282,94.250926,-0.730133,-0.611111,-0.880907,0,Cross-sell complementary products and provide ...,Retention / Cross-sell,Medium
3,12349.0,2,New / Recently Acquired,Onboarding and second-purchase conversion,19,1,1457.55,0,54.025694,0.000000,0.000000,0.000000,1,Welcome journey with product discovery and sec...,New Customer Onboarding,High
4,12350.0,0,Inactive / Declining,Reactivation,310,1,294.40,0,54.025694,-1.000000,-1.000000,-1.000000,0,Win-back campaign with personalized offers bas...,Reactivation,High
5,12352.0,1,Established Active,Retention and relationship growth,36,7,1385.74,260,43.347685,0.353476,-0.125000,0.297858,0,Cross-sell complementary products and provide ...,Retention / Cross-sell,Medium
6,12353.0,0,Inactive / Declining,Reactivation,204,1,89.00,0,54.025694,-1.000000,-1.000000,-1.000000,0,Win-back campaign with personalized offers bas...,Reactivation,High
7,12354.0,0,Inactive / Declining,Reactivation,232,1,1079.40,0,54.025694,-1.000000,-1.000000,-1.000000,0,Win-back campaign with personalized offers bas...,Reactivation,High
8,12355.0,0,Inactive / Declining,Reactivation,214,1,459.40,0,54.025694,-1.000000,-1.000000,-1.000000,0,Win-back campaign with personalized offers bas...,Reactivation,High
9,12356.0,1,Established Active,Retention and relationship growth,23,3,2487.43,302,151.475694,-0.971975,-0.416667,-0.996280,0,Cross-sell complementary products and provide ...,Retention / Cross-sell,Medium


In [64]:
# ============================================================
# PHASE 2 — RECOMMENDATION VALIDATION
# Cell 33
# ============================================================

print("=" * 70)
print("RECOMMENDATION ENGINE VALIDATION")
print("=" * 70)

print("\nRecommendation counts:")
print(
    final_customer_segments["Recommendation"]
    .value_counts()
)

print("\nCampaign types:")
print(
    final_customer_segments["CampaignType"]
    .value_counts()
)

print("\nPriority distribution:")
print(
    final_customer_segments["Priority"]
    .value_counts()
)

print("\nMissing recommendation fields:")
print(
    final_customer_segments[
        ["Recommendation", "CampaignType", "Priority"]
    ].isna().sum()
)

RECOMMENDATION ENGINE VALIDATION

Recommendation counts:
Recommendation
Welcome journey with product discovery and second-purchase incentive              1374
Cross-sell complementary products and provide loyalty-oriented offers             1299
Win-back campaign with personalized offers based on previous purchases             973
Premium personalized recommendations, loyalty benefits and early-access offers     689
Name: count, dtype: int64

Campaign types:
CampaignType
New Customer Onboarding    1374
Retention / Cross-sell     1299
Reactivation                973
High-Value Retention        689
Name: count, dtype: int64

Priority distribution:
Priority
High      3036
Medium    1299
Name: count, dtype: int64

Missing recommendation fields:
Recommendation    0
CampaignType      0
Priority          0
dtype: int64


In [65]:
# Save final customer-level segmentation with recommendations

output_path = "../results/phase2/phase2_customer_segments.csv"

final_customer_segments.to_csv(
    output_path,
    index=False
)

print("=" * 70)
print("FINAL CUSTOMER SEGMENTS SAVED")
print("=" * 70)

print("Shape:", final_customer_segments.shape)
print("File:", output_path)

print("\nColumns:")
print(final_customer_segments.columns.tolist())

FINAL CUSTOMER SEGMENTS SAVED
Shape: (4335, 16)
File: ../results/phase2/phase2_customer_segments.csv

Columns:
['CustomerID', 'Cluster', 'Status', 'MarketingObjective', 'Recency', 'Frequency', 'Monetary', 'CustomerLifespanDays', 'AveragePurchaseIntervalDays', 'SpendingTrend', 'FrequencyTrend', 'QuantityTrend', 'NewCustomer', 'Recommendation', 'CampaignType', 'Priority']


In [66]:
print("=" * 70)
print("FINAL PHASE 2 VALIDATION")
print("=" * 70)

print("\n1. Dataset shape:")
print(final_customer_segments.shape)

print("\n2. Duplicate CustomerIDs:")
print(final_customer_segments["CustomerID"].duplicated().sum())

print("\n3. Missing values:")
print(final_customer_segments.isna().sum())

print("\n4. Cluster distribution:")
print(final_customer_segments["Cluster"].value_counts().sort_index())

print("\n5. Cluster percentages:")
print(
    (final_customer_segments["Cluster"].value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

print("\n6. Recommendation distribution:")
print(final_customer_segments["CampaignType"].value_counts())

print("\n7. Priority distribution:")
print(final_customer_segments["Priority"].value_counts())

print("\n8. Unique customers:")
print(final_customer_segments["CustomerID"].nunique())

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

FINAL PHASE 2 VALIDATION

1. Dataset shape:
(4335, 16)

2. Duplicate CustomerIDs:
0

3. Missing values:
CustomerID                     0
Cluster                        0
Status                         0
MarketingObjective             0
Recency                        0
Frequency                      0
Monetary                       0
CustomerLifespanDays           0
AveragePurchaseIntervalDays    0
SpendingTrend                  0
FrequencyTrend                 0
QuantityTrend                  0
NewCustomer                    0
Recommendation                 0
CampaignType                   0
Priority                       0
dtype: int64

4. Cluster distribution:
Cluster
0     973
1    1299
2    1374
3     689
Name: count, dtype: int64

5. Cluster percentages:
Cluster
0    22.45
1    29.97
2    31.70
3    15.89
Name: proportion, dtype: float64

6. Recommendation distribution:
CampaignType
New Customer Onboarding    1374
Retention / Cross-sell     1299
Reactivation                973
Hig